# Gateway Auth와 Okta 통합

AgentCore Identity를 사용하면 AgentCore Runtime의 에이전트나 도구를 호출하는 사용자 및 애플리케이션의 인바운드 액세스(Inbound Auth)를 검증하거나 AgentCore Gateway target에 대한 액세스를 검증할 수 있습니다. 또한 에이전트에서 외부 서비스 또는 Gateway target으로 향하는 안전한 아웃바운드 액세스(Outbound Auth)를 제공합니다. 기존 identity provider(예: Amazon Cognito)와 통합되며, 독립적으로 또는 OAuth를 통해 사용자를 대신하여 작동하는 에이전트에 권한 경계를 적용합니다.

Inbound Auth는 AgentCore Runtime, AgentCore Gateway 또는 다른 환경에 호스팅된 에이전트나 도구를 호출하려는 요청자를 검증합니다. Inbound Auth는 IAM(SigV4 credentials) 또는 OAuth 권한 부여와 함께 작동합니다.

Amazon Bedrock AgentCore는 기본적으로 IAM credentials를 사용하므로 에이전트에 대한 사용자 요청은 사용자의 IAM credentials로 인증됩니다. 이 튜토리얼에서는 Okta IdP와 OAuth를 사용하므로 AgentCore Runtime 리소스 또는 AgentCore Gateway endpoint를 구성할 때 다음 항목을 지정해야 합니다.

- OAuth discovery server URL: OpenID Connect discovery URL의 `^.+/.well-known/openid-configuration$` pattern과 일치해야 하는 문자열

- Allowed audiences: JWT token에 허용할 audience 목록

- Allowed clients: 허용할 client identifier 목록

AgentCore CLI를 사용하는 경우 `configure` 명령으로 AgentCore Runtime의 권한 부여 유형과 OAuth discovery server를 지정할 수 있습니다. `CreateAgentRuntime` operation이나 Amazon Bedrock AgentCore console도 사용할 수 있습니다. Gateway를 생성할 때는 `CreateGateway` operation 또는 console을 사용합니다.

사용자가 에이전트를 사용하려면 먼저 client application에서 OAuth authorizer를 통해 사용자를 인증해야 합니다. client는 bearer token을 받은 뒤 에이전트 호출 요청에 포함합니다. 에이전트는 token을 받으면 액세스를 허용하기 전에 authorization server로 token을 검증합니다.

## 개요

이 튜토리얼에서는 Okta를 identity provider로 사용하여 Inbound Auth를 구성합니다. 사용자 한 명과 app client 하나가 포함된 Okta tenant를 설정합니다. 그리고 Okta app client 기반 Inbound Auth를 적용한 Amazon Bedrock AgentCore Runtime에 기존 에이전트를 호스팅하는 방법을 알아봅니다. 에이전트가 연동할 Amazon Bedrock AgentCore Gateway에도 Okta 기반 Inbound Auth를 설정합니다.

### 튜토리얼 아키텍처

<figure>
    <img src="images/16.png">
</figure>

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                           |
| 에이전트 유형       | 단일                                                                             |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 4                                                        |
| 튜토리얼 구성 요소  | AgentCore Runtime에 에이전트 호스팅, Strands Agent 및 Amazon Bedrock 모델 사용  |
| 튜토리얼 분야       | 산업 공통                                                                        |
| 예제 난이도         | 쉬움                                                                             |
| Inbound Auth        | Okta                                                                             |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                                     |


### 주요 기능

* Okta 기반 Inbound Auth 및 Outbound Auth를 적용해 Amazon Bedrock AgentCore Runtime에 에이전트 호스팅
* Okta 기반 Inbound Auth를 적용해 Amazon Bedrock AgentCore Gateway 호스팅
* Amazon Bedrock 모델 사용
* Strands Agents 사용

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* 새 IAM 역할, 정책, 사용자를 생성할 IAM 권한
* 새 AgentCore Agent를 생성할 IAM 권한
* Okta 계정
* Amazon Bedrock AgentCore SDK
* Strands Agents
* 실행 중인 Docker

## Okta IdP 설정

App client 하나와 테스트 사용자 한 명이 포함된 Okta demo tenant를 설정합니다. 이 튜토리얼의 뒷부분에서 배포할 에이전트를 호출할 때 Okta에서 제공하는 JWT token을 사용합니다. 이미 Okta 계정이 있다면 해당 계정으로 로그인합니다.

https://developer.okta.com/signup/ 으로 이동하여 **Sign up for Integrator Free Plan**을 선택하고 가입합니다.

1. 계정에 로그인합니다.<br><br>
2. **Directory**, **People**을 차례로 선택한 뒤 **Add person**을 클릭합니다.
    <figure>
        <img src="images/9.png">
    </figure><br><br>
3. 양식을 작성합니다.
    <ol type="1">
        <li><b>Activation</b>에서 <b>Activate now</b>를 선택합니다.</li>
        <li><b>I will set password</b> 확인란을 선택하고 사용자의 password를 설정합니다.</li>
        <li><b>User must change password on first login</b> 확인란을 선택 해제합니다.</li>
        <li><b>Save</b>를 클릭합니다.</li>
    </ol>

    <figure>
        <img src="images/10.png">
    </figure>
    <br><br>
4. **Applications**를 선택한 뒤 **Create App Integration**을 클릭합니다.
    <figure>
        <img src="images/1.png">
    </figure>
    <br><br>

5. sign-in method로 **OIDC - OpenID Connect**를 선택하고 application type으로 **Web Application**을 선택한 뒤 **Next**를 클릭합니다.

    <figure>
        <img src="images/2.png">
    </figure><br><br>
    <ol type="a">
        <li>App integration name에 <b>Travel Assistant</b>를 입력하고 <b>Proof of possession</b>은 선택하지 않은 상태로 둡니다. grant type으로 <b>Authorization Code</b>를 선택합니다. 
        <br><br>
        <figure>
            <img src="images/3.png">
        </figure>
        <br><br>
        </li>
        <li>sign-in URI에 <b>http://127.0.0.1:5000/callback</b>과 <b>https://bedrock-agentcore.us-west-2.amazonaws.com/identities/oauth2/callback</b>을 추가합니다. sign-out redirect URI는 그대로 둡니다.
        <br><br>
        <figure>
            <img src="images/4.png">
        </figure>
        <br><br>
        </li>
        <li>assignments에서 <b>Allow everyone in your organization to access</b>를 선택하고 <b>Enable immediate access</b>는 선택된 상태로 둡니다. 그런 다음 <b>Save</b>를 클릭합니다.
        <br><br>
        <figure>
            <img src="images/5.png">
        </figure>
        <br><br>
        </li>
        <li>나중에 사용할 <b>Client ID</b>와 <b>Secret</b>을 복사합니다.</li>
        <br><br>
        <figure>
            <img src="images/6.png">
        </figure>
        <br><br>
    </ol><br>

6. 왼쪽 메뉴에서 **Security**, **API**를 차례로 선택한 뒤 authorization server의 이름을 클릭합니다.
    <figure>
        <img src="images/17.png">
    </figure><br><br>
    <ol type="a">
        <li><b>Audience</b>를 복사하여 나중에 사용할 수 있도록 저장합니다.</li>
            <ul>
                <li><b>참고</b>: 이 예제에서는 기본 <b>Audience</b>를 변경했습니다. 다른 app에 영향을 주지 않도록 audience를 변경하려면 새 authorization server를 추가하는 것이 좋습니다.</li><br>
            </ul>
        <figure>
            <img src="images/7.png">
        </figure>
        <br><br>
        <li><b>Scopes</b>, <b>Add Scope</b>를 차례로 클릭합니다.</li><br>
        <figure>
            <img src="images/43.png">
        </figure>
        <br><br>
        <li>name에 <b>okta.myAccount.read</b>를 사용하고 <b>Display Phrase</b>와 <b>Description</b>을 입력합니다.</li>
        <li><b>User Consent</b>를 <b>implicit</b>으로 설정합니다.</li>
        <li><b>Block services</b>, <b>Default scope</b>, <b>Metadata</b>는 기본값으로 둡니다.</li>
        <li><b>Save</b>를 클릭합니다.
        <figure>
            <img src="images/44.png">
        </figure><br>
        <li><b>Claims</b>를 클릭하고 다음 <b>client_id</b> 및 <b>scope</b> claim을 추가합니다.</li><br>
        <figure>
            <img src="images/8.png">
        </figure><br>
        <li><b>Access Policies</b>를 클릭한 뒤 <b>Add New Access Policy</b>를 클릭합니다.</li><br>
        <figure>
            <img src="images/39.png">
        </figure><br>
        <li><b>Name</b>과 <b>Description</b>을 입력하고 <b>Create Policy</b>를 클릭합니다.</li><br>
        <figure>
            <img src="images/40.png">
        </figure><br>
        <li><b>Add rule</b>을 클릭합니다.</li><br>
        <figure>
            <img src="images/41.png">
        </figure><br>
        <li>rule에 <b>Rule Name</b>을 지정하고 <b>Create Rule</b>을 클릭합니다.</li><br>
        <figure>
            <img src="images/42.png">
        </figure><br>
    </ol>




## API Gateway 생성

이 CloudFormation template은 JWT 인증을 사용하는 serverless API를 생성합니다.
* Okta token을 검증하는 JWT authorizer가 포함된 Amazon API Gateway
* 유효한 JWT 인증이 필요한 보호된 **GET /travel-plans** endpoint
* **GET /travel-plans** endpoint를 호출하면 실행되는 AWS Lambda 함수

AgentCore Gateway는 MCP 요청을 기반 API Gateway endpoint에 대한 HTTP 호출로 변환하는 protocol adapter 역할을 하여 이 Amazon API Gateway를 MCP(Model Context Protocol) server로 전환합니다.

아래 코드 셀을 실행하여 CloudFormation template을 로컬 파일 시스템에 **template.yaml**로 저장합니다. 

In [ ]:
%%writefile template.yaml
AWSTemplateFormatVersion: '2010-09-09'
Parameters:
  JwtIssuerUrl:
    Type: String
    Description: The URL of the JWT issuer (e.g., Cognito user pool URL).
    MinLength: 10 # 선택 사항: 최소 길이와 같은 제약 조건을 추가할 수 있음
    MaxLength: 200 # 선택 사항: 최대 길이
    # 필요한 경우 Default 값을 추가할 수도 있음

  JwtAudienceList:
    Type: CommaDelimitedList # <-- 여러 audience에 CommaDelimitedList 사용
    Description: A comma-separated list of expected audience(s) for the JWT (e.g.,
      "my-api-audience-1,my-api-audience-2").
    # 선택 사항: 필요한 경우 Default 또는 다른 제약 조건을 추가할 수 있음
    # 기본값: "my-default-audience"

Resources:
  MyHttpApi:
    Type: AWS::ApiGatewayV2::Api
    Properties:
      Name: MyHttpApi
      ProtocolType: HTTP

  MyJwtAuthorizer:
    Type: AWS::ApiGatewayV2::Authorizer
    Properties:
      ApiId: !Ref MyHttpApi
      AuthorizerType: JWT
      IdentitySource:
        - $request.header.Authorization
      JwtConfiguration:
        Audience: !Ref JwtAudienceList
        Issuer: !Ref JwtIssuerUrl
      Name: MyJwtAuthorizer

  MyLambdaFunction:
    Type: AWS::Lambda::Function
    Properties:
      FunctionName: MyLambdaFunction
      Handler: lambda_function.lambda_handler
      Runtime: python3.12
      Code:
        ZipFile: |
          exports.handler = async (event) => {
              // non-proxy integration에서 Lambda 함수는 전체 HTTP 요청이 아니라
              // API Gateway에서 매핑된 입력을 수신함
              console.log('Received event:', JSON.stringify(event, null, 2));

              const name = event.name || "World";
              const message = `Hello, ${name}! (from non-proxy integration)`;

              // non-proxy integration에서는 간단한 문자열이나 객체 등을 반환할 수 있음
              // 그러면 API Gateway가 mapping template을 사용해 적절한 HTTP 응답으로 형식을 지정함
              return { "message": message }; // 간단한 JSON 응답 예제
          };
      MemorySize: 128
      Timeout: 30
      Role: !GetAtt MyLambdaExecutionRole.Arn

  MyLambdaExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service:
                - lambda.amazonaws.com
            Action:
              - sts:AssumeRole
      Policies:
        - PolicyName: MyLambdaPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: arn:aws:logs:*:*:*

  MyApiIntegration:
    Type: AWS::ApiGatewayV2::Integration
    Properties:
      PayloadFormatVersion: "2.0"
      ApiId: !Ref MyHttpApi
      IntegrationType: AWS_PROXY
      IntegrationUri: !Join
        - ''
        - - 'arn:'
          - !Ref 'AWS::Partition'
          - ':apigateway:'
          - !Ref 'AWS::Region'
          - ':lambda:path/2015-03-31/functions/'
          - !GetAtt MyLambdaFunction.Arn
          - /invocations
      IntegrationMethod: POST # Lambda 호출은 일반적으로 POST를 사용함
      # IntegrationRequest 및 IntegrationResponse 섹션에 표시된 것처럼
      # AWS integration type에 대한 mapping template도 정의해야 함

  MyApiRoute:
    Type: AWS::ApiGatewayV2::Route
    Properties:
      ApiId: !Ref MyHttpApi
      RouteKey: GET /travel-plans # IntegrationMethod에 맞게 POST로 변경됨
      Target: !Join
        - /
        - - integrations
          - !Ref MyApiIntegration
      AuthorizationType: JWT
      AuthorizerId: !Ref MyJwtAuthorizer

1. us-west-2 리전의 CloudFormation으로 이동하여 **Create stack**을 클릭합니다.
2. **With new resources(standard)**를 선택합니다.
<figure>
    <img src="images/20.png">
</figure>

3. prerequisite에서 **Choose an existing template**을 선택합니다.
4. specify template에서 **Upload a template file**을 선택하고 이전 단계에서 저장한 **template.yaml** 파일을 선택합니다.
5. **Next**를 클릭합니다.

<figure>
    <img src="images/21.png">
</figure>

6. **stack name**을 입력합니다(예: test-deployment-stack).
7. JwtAudienceList parameter에 **Audience**를 입력합니다. 
8. JwtIssueURL에 **Issuer URL**을 입력합니다(예: https://{yoursubdomain}.okta.com/oauth2/default).

<figure>
    <img src="images/22.png">
</figure>

9. **I acknowledge that AWS CloudFormation might create IAM resources** 확인란을 선택합니다.
10. **Next**를 클릭합니다.
<figure>
    <img src="images/32.png">
</figure>

11. **Submit**을 클릭합니다.

<figure>
    <img src="images/49.png">
</figure>

## Lambda 함수 업데이트

1. **Lambda** > **Functions**로 이동한 뒤 **MyLambdaFunction**을 클릭합니다.
<figure>
    <img src="images/23.png">
</figure>

2. 아래 source code를 **Code** 섹션에 복사합니다.
3. <b>index.js</b> 파일의 이름을 <b>lambda_function.py</b>로 변경합니다.
4. **Deploy**를 클릭합니다. 
<figure>
    <img src="images/24.png">
</figure>

5. **API Gateway** > **Integrations**로 이동하여 **Manage integrations**를 클릭합니다.
<figure>
    <img src="images/56.png">
</figure>

6. **Integration details**에서 **Edit**을 클릭합니다.
<figure>
    <img src="images/57.png">
</figure>

7. 최신 **Lambda function** ARN으로 업데이트하고 **Save**를 클릭합니다.
<figure>
    <img src="images/58.png">
</figure>

AWS Lambda 코드:

In [ ]:
import json

# Mock 데이터 저장소(프로덕션에서는 데이터베이스 사용)
MOCK_TRAVEL_PLANS = [
    {
        "id": "plan-001",
        "user_id": "user-123",
        "email": "john.doe@example.com",
        "destination": "Paris, France",
        "departure_date": "2024-03-15",
        "return_date": "2024-03-22",
        "accommodation": "Hotel Le Marais",
        "activities": ["Eiffel Tower", "Louvre Museum", "Seine River Cruise"],
        "budget": 2500.00,
        "status": "confirmed",
    },
    {
        "id": "plan-002",
        "user_id": "user-123",
        "email": "john.doe@example.com",
        "destination": "Tokyo, Japan",
        "departure_date": "2024-05-10",
        "return_date": "2024-05-20",
        "accommodation": "Tokyo Grand Hotel",
        "activities": ["Mount Fuji", "Sensoji Temple", "Shibuya Crossing"],
        "budget": 3500.00,
        "status": "planned",
    },
    {
        "id": "plan-003",
        "user_id": "user-456",
        "email": "jane.smith@example.com",
        "destination": "New York, USA",
        "departure_date": "2024-04-01",
        "return_date": "2024-04-07",
        "accommodation": "Manhattan Plaza Hotel",
        "activities": ["Statue of Liberty", "Central Park", "Broadway Show"],
        "budget": 2000.00,
        "status": "confirmed",
    },
    {
        "id": "plan-004",
        "user_id": "user-456",
        "email": "jane.smith@example.com",
        "destination": "Barcelona, Spain",
        "departure_date": "2024-06-15",
        "return_date": "2024-06-25",
        "accommodation": "Barcelona Beach Resort",
        "activities": ["Sagrada Familia", "Park Güell", "Las Ramblas"],
        "budget": 2800.00,
        "status": "planned",
    },
    {
        "id": "plan-005",
        "user_id": "user-789",
        "email": "bob.wilson@example.com",
        "destination": "Sydney, Australia",
        "departure_date": "2024-07-20",
        "return_date": "2024-08-03",
        "accommodation": "Sydney Harbour Hotel",
        "activities": ["Opera House", "Harbour Bridge", "Bondi Beach"],
        "budget": 4500.00,
        "status": "tentative",
    },
]


def lambda_handler(event, context):
    """
    여행 계획 API의 기본 Lambda 핸들러입니다.
    """
    query_parameters = event.get("queryStringParameters", {})
    try:
        # HTTP method와 path를 기준으로 route 지정
        return get_travel_plans(query_parameters)
    except Exception as e:
        return create_response(500, {"error": "Internal Server Error", "message": str(e)})


def create_response(status_code, body):
    """
    API Gateway Lambda 응답을 생성합니다.
    """
    return {"statusCode": status_code, "body": json.dumps(body)}


def get_travel_plans(query_params):
    """
    user_id 또는 email로 여행 계획을 조회합니다.
    쿼리 매개변수:
    - user_id: 사용자 ID로 필터링
    - email: 이메일 주소로 필터링
    """
    user_id = query_params.get("user_id") if query_params else None
    email = query_params.get("email") if query_params else None

    # 하나 이상의 parameter가 제공되었는지 검증
    if not user_id and not email:
        return create_response(
            400,
            {
                "error": "Either user_id or email must be provided",
                "message": "Please provide user_id or email as query parameter",
            },
        )

    # 제공된 parameter를 기준으로 여행 계획 필터링
    filtered_plans = []

    for plan in MOCK_TRAVEL_PLANS:
        if user_id and plan["user_id"] == user_id:
            filtered_plans.append(plan)
        elif email and plan["email"].lower() == email.lower():
            filtered_plans.append(plan)

    # 출발일 기준 정렬(최신순)
    filtered_plans.sort(key=lambda x: x.get("departure_date", ""), reverse=True)

    # 응답 반환
    if filtered_plans:
        return create_response(
            200,
            {
                "success": True,
                "count": len(filtered_plans),
                "travel_plans": filtered_plans,
                "filter": {"user_id": user_id, "email": email},
            },
        )
    else:
        return create_response(
            404,
            {
                "success": False,
                "message": "No travel plans found for the specified user",
                "filter": {"user_id": user_id, "email": email},
                "travel_plans": [],
            },
        )

## Amazon Bedrock AgentCore Gateway 생성

아래 코드는 **demo_openapi.yaml** 파일을 로컬 저장 장치에 기록합니다. 이 파일은 이전 단계에서 생성한 API Gateway로 요청을 proxy하는 Amazon Bedrock AgentCore Gateway를 생성하는 데 사용됩니다.
<ol type="1">
<li><b>API Gateway</b> > <b>APIs</b> > <b>Stages</b>로 이동하여 <b>Create</b>를 클릭합니다.</li>
<figure>
    <img src="images/50.png">
</figure>
<li>stage <b>Name</b>을 <b>default</b>로 설정합니다.</li>
<li><b>Enable automatic deployment</b>를 선택합니다.</li>
<li><b>Create</b>를 클릭합니다.</li>
<figure>
    <img src="images/51.png">
</figure>
<li><b>Default endpoint</b>를 기록합니다.</li>
<figure>
    <img src="images/52.png">
</figure>
<li>아래 코드 셀을 실행하여 <b>demo_openapi.yaml</b> 파일을 로컬 저장 장치에 저장합니다.</li>
    <ol type="a">
            <li><b>demo_oepnapi.yaml</b>을 열고 <b>server URL</b>의 <b>{yoursubdomain}</b>을 이전 단계의 <b>Invoke ULR</b>과 일치하도록 수정합니다.</li>
            <li><b>authorization URL</b>과 <b>token URL</b>의 <b>{yoursubdomain}</b>을 Okta tenant와 일치하도록 수정한 뒤 파일을 저장합니다. 이 값은 Okta tenant에 로그인할 때 사용하는 URL에서 확인할 수 있습니다.</li>
    </ol>
</ol>

In [ ]:
%%writefile demo_openapi.yaml
openapi: 3.0.0
info:
  title: Travel Plans API
  description: API for retrieving user travel plans with secure authentication
  version: 1.0.0
  contact:
    name: Travel Plans Developer
    email: developer@example.com
  license:
    name: Apache 2.0
    url: http://www.apache.org/licenses/LICENSE-2.0.html

servers:
  - url: https://{yoursubdomain}.execute-api.us-west-2.amazonaws.com/default
    description: Production server

paths:
  /travel-plans:
    get:
      summary: Retrieve Travel Plans
      description: Fetch travel plans by user ID or email
      operationId: getTravelPlans
      security:
        - OAuth2:
          - read:travel-plans
        - ApiKeyAuth: []
      parameters:
        - in: query
          name: user_id
          schema:
            type: string
          required: false
          description: Unique identifier of the user
          example: "user-123"
        
        - in: query
          name: email
          schema:
            type: string
            format: email
          required: false
          description: Email address of the user
          example: "john.doe@example.com"
      
      responses:
        '200':
          description: Successful retrieval of travel plans
          content:
            application/json:
              schema:
                type: object
                properties:
                  success:
                    type: boolean
                  count:
                    type: integer
                  travel_plans:
                    type: array
                    items:
                      $ref: '#/components/schemas/TravelPlan'
                  filter:
                    type: object
                    properties:
                      user_id:
                        type: string
                      email:
                        type: string
        
        '400':
          description: Bad Request - Missing query parameters
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'
        
        '401':
          description: Unauthorized - Invalid or missing authentication
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'
        
        '403':
          description: Forbidden - Insufficient permissions
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'
        
        '404':
          description: No travel plans found
          content:
            application/json:
              schema:
                type: object
                properties:
                  success:
                    type: boolean
                  message:
                    type: string
                  filter:
                    type: object
                    properties:
                      user_id:
                        type: string
                      email:
                        type: string
                  travel_plans:
                    type: array
                    items: {}
        
        '500':
          description: Internal Server Error
          content:
            application/json:
              schema:
                $ref: '#/components/schemas/ErrorResponse'

components:
  securitySchemes:
    OAuth2:
      type: oauth2
      flows:
        authorizationCode:
          authorizationUrl: https://{yoursubdomain}.okta.com/oauth2/default/v1/authorize
          tokenUrl: https://{yoursubdomain}.okta.com/oauth2/default/v1/token
          scopes:
            read:travel-plans: Read access to travel plans
            write:travel-plans: Write access to travel plans
            delete:travel-plans: Delete access to travel plans
  
    ApiKeyAuth:
      type: apiKey
      in: header
      name: X-API-Key

    BearerAuth:
      type: http
      scheme: bearer
      bearerFormat: JWT

  schemas:
    TravelPlan:
      type: object
      required:
        - id
        - user_id
        - email
        - destination
        - departure_date
        - return_date
        - status
      properties:
        id:
          type: string
          description: Unique identifier for the travel plan
          example: "plan-001"
        user_id:
          type: string
          description: Unique identifier of the user
          example: "user-123"
        email:
          type: string
          format: email
          description: Users email address
          example: "john.doe@example.com"
        destination:
          type: string
          description: Travel destination
          example: "Paris, France"
        departure_date:
          type: string
          format: date
          description: Date of departure
          example: "2024-03-15"
        return_date:
          type: string
          format: date
          description: Date of return
          example: "2024-03-22"
        accommodation:
          type: string
          description: Name of accommodation
          example: "Hotel Le Marais"
        activities:
          type: array
          items:
            type: string
          description: List of planned activities
          example: ["Eiffel Tower", "Louvre Museum"]
        budget:
          type: number
          format: float
          description: Estimated travel budget
          example: 2500.00
        status:
          type: string
          description: Status of the travel plan
          enum:
            - confirmed
            - planned
            - tentative
          example: "confirmed"

    ErrorResponse:
      type: object
      properties:
        error:
          type: string
        message:
          type: string
        error_code:
          type: string
        timestamp:
          type: string
          format: date-time

    OAuthToken:
      type: object
      properties:
        access_token:
          type: string
          description: JWT access token
        token_type:
          type: string
          enum:
            - Bearer
        expires_in:
          type: integer
          description: Token expiration time in seconds
        refresh_token:
          type: string
          description: Token to obtain a new access token

tags:
  - name: Travel Plans
    description: Operations related to travel plan retrieval
  - name: Authentication
    description: OAuth2 and API Key authentication methods

x-security-definitions:
  - name: OAuth2
    description: >
      OAuth 2.0 Authentication:
      - Authorization Code Flow
  - name: API Key
    description: >
      API Key authentication for service-to-service communication

x-rate-limiting:
  limit: 100
  period: 1 minute
  
x-error-handling:
  generic-errors:
    - 400: Bad Request
    - 401: Unauthorized
    - 403: Forbidden
    - 404: Not Found
    - 500: Internal Server Error

2. **demo_openapi.yaml**을 S3 bucket에 업로드합니다. 사용할 bucket은 자유롭게 선택할 수 있습니다.
<figure>
    <img src="images/25.png">
</figure>

3. **Amazon Bedrock AgentCore**로 이동합니다. 
4. 왼쪽 메뉴에서 **Identity**를 선택합니다.
5. **Add OAuth client / API key**, **Add Oauth client**를 차례로 클릭합니다.
<figure>
    <img src="images/26.png">
</figure>

6. Provider에서 **Custom provider**를 선택합니다.
7. **Discovery URL**을 선택하고 **Client ID**, **Client secret**, **Discovery URL**(https://{yoursubdomain}.okta.com/oauth2/default/.well-known/openid-configuration)을 입력한 뒤 나중에 사용할 resource provider 이름을 저장합니다.
8. **Add OAuth Client**를 클릭합니다.
<figure>
    <img src="images/27.png">
</figure>


9. 왼쪽 메뉴에서 **Gateways**를 선택한 뒤 **Create Gateway**를 클릭합니다.
<figure>
    <img src="images/28.png">
</figure>

10. **Gateway name**을 설정하거나 기본 이름을 사용합니다.
<figure>
    <img src="images/29.png">
</figure>

11. **Inbound Auth configurations**에서 **Use existing Identity provider configurations**를 선택합니다.
12. **Discovery URL**(https://{yoursubdomain}.okta.com/oauth2/default/.well-known/openid-configuration), **Allowed audiences**, **Allowed clients**를 입력합니다. Allowed audiences와 Allowed clients에는 Okta tenant의 audience와 Client ID를 사용합니다.
<figure>
    <img src="images/30.png">
</figure>

13. Target:{target name}에서 target type으로 **REST API**를 선택합니다.
14. REST API type으로 **OpenAPI schema**를 선택합니다.
15. OpenAPI schema에서 **Define with an S3 resource**를 선택합니다.
16. **2단계**에서 업로드한 OpenAPI 문서의 위치로 이동하여 해당 문서를 선택합니다.
17. Outbound Auth configurations에서 **OAuth client**를 선택합니다.
18. OAuth client에서 **3~5단계**에 생성한 Amazon Bedrock AgentCore Identity를 선택합니다.
<figure>
    <img src="images/31.png">
</figure>

19. Scopes에 <b>okta.myAccount.read</b>를 추가합니다.
20. **Save**를 클릭합니다.
<figure>
    <img src="images/33.png">
</figure>



 
## AgentCore Runtime 배포를 위한 에이전트 준비

이 코드는 Strands framework와 Bedrock AgentCore SDK를 사용하는 **Travel Assistant chatbot**을 정의합니다.

Amazon Bedrock AgentCore Gateway를 통해 customer ID 또는 email로 기존 여행 계획을 가져올 수 있는 travel assistant agent를 설정합니다. `@requires_access_token()` decorator는 access token을 가져와 decorated function에 주입하는 과정을 자동으로 관리하여 OAuth 2.0 인증을 처리합니다. 이전 단계에서 생성한 outbound OAuth client를 사용해 Okta 인증 흐름을 구성하고 필요한 scope(`okta.myAccount.read`)를 요청합니다. 인증이 필요하면 사용자가 방문하여 애플리케이션을 승인할 수 있도록 authorization URL을 console에 출력합니다. 사용자가 OAuth 흐름을 완료하면 decorator가 발급된 access token을 `need_token_3LO_async` 함수의 parameter로 자동 주입하고 token vault에 저장하여 반복적인 인증 요청을 방지합니다.

1. 아래 셀을 실행하여 에이전트 코드를 로컬 파일 시스템에 **my_agent_mcp.py**로 저장합니다.
2. <b>Amazon AgentCore Gateway</b> > <b>Gateways</b>로 이동하여 사용할 Gateway를 클릭합니다(예: gateway-quick-start-234a1).
<figure>
    <img src="images/53.png">
</figure>

3. 파일에서 `gateway_url`의 **{yoursubdomain}**을 **Gateway resource URL**과 일치하도록 바꿉니다.
<figure>
    <img src="images/38.png">
</figure>

4. `provider_name`의 **{yourprovidername}**을 outbound OAuth client 이름으로 바꿉니다(예: resource-provider-oauth-client-1wbak).
5. 파일을 저장합니다.

In [ ]:
%%writefile my_agent_mcp.py
import json
import requests
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent, tool
from strands.tools.mcp import MCPClient
from strands_tools import calculator, current_time

# AgentCore SDK 가져오기
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_access_token

WELCOME_MESSAGE = """
Welcome to the Travel Assistant! How can I help you today?
"""

SYSTEM_PROMPT = """
You are an helpful travel support assistant.
When provided with a customer email, gather all necessary info and prepare the response.
When asked about existing travel plans, look for it and customize the summary based on the prompt.
Don't mention the customer ID in your reply.
"""

# 전역 token 저장소
okta_access_token = None

def create_streamable_http_transport(mcp_url: str, access_token: str):
       return streamablehttp_client(mcp_url, headers={"Authorization": f"Bearer {access_token}"})

# AgentCore app 생성
app = BedrockAgentCoreApp()

async def agent_task(user_message: str) -> None:

    global okta_access_token
    okta_access_token = await need_token_3LO_async(access_token='')
    response = ''

    gateway_url = "https://{yoursubdomain}.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp"
    mcp_client = MCPClient(lambda: create_streamable_http_transport(gateway_url , okta_access_token))

    with mcp_client:
        #tools = await get_full_tools_list(mcp_client)
        #print(f"Found the following tools: {[tool.tool_name for tool in tools]}")
    
        agent = Agent(tools=mcp_client.list_tools_sync())
        response = agent(user_message)
        print(response)
    
    return response.message['content'][0]['text']

# Okta Access Token 주입
@requires_access_token(# 위에서 생성한 것과 동일한 credential provider 이름 사용
    provider_name= "{yourprovidername}",
    # MCP Server에 액세스하는 데 필요한 Okta OAuth2 scope
    scopes= ["okta.myAccount.read"],
    # OAuth 2.0 Authorization Code flow로 설정
    auth_flow= "USER_FEDERATION",
    # authorization URL을 console에 출력
    on_auth_url= lambda x: print("\nPlease copy and paste this URL in your browser:\n" + x),
    # false이면 발급된 access token을 cache에 저장
    force_authentication= False,) 
async def need_token_3LO_async(*, access_token: str) -> str:
    """OAuth 인증 흐름을 처리합니다."""
    global okta_access_token
    okta_access_token = access_token
    return access_token


# 에이전트를 호출하는 entry point 함수 지정
@app.entrypoint
async def invoke(payload):
    """에이전트 호출을 처리합니다."""
    user_message = payload.get(
        "prompt", "No prompt found in input, please guide customer to create a json payload with prompt key"
    )

    result = await agent_task(user_message)
    return result

if __name__ == "__main__":
    app.run()

이 코드를 실행하려면 Python 환경에 Strands Agents module이 설치되어 있어야 합니다.

dependency를 설치할 가상 환경을 생성하고 활성화합니다.

In [ ]:
!python -m venv .venv 
!source .venv/bin/activate

Strands Agents module, AgentCore SDK, AgentCore starter toolkit을 dependency 파일에 추가하고 **requirements.txt**로 저장합니다.

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore
bedrock-agentcore-starter-toolkit


그런 다음 가상 환경에 모든 requirements를 설치합니다.

In [ ]:
%pip install -r requirements.txt

## AgentCore Runtime에 에이전트 배포
`CreateAgentRuntime` operation은 container image, 환경 변수, 암호화 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. protocol 설정(HTTP, MCP)과 권한 부여 메커니즘을 구성하여 client가 에이전트와 통신하는 방식도 제어할 수 있습니다.

참고: 운영 환경에서는 코드를 container로 package하고 CI/CD pipeline과 IaC를 사용하여 ECR에 push하는 것이 좋습니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 artifact를 간편하게 package하고 AgentCore Runtime에 배포합니다.

### AgentCore Runtime 배포를 위한 에이전트 구성
이제 starter toolkit을 사용하여 entrypoint, 앞에서 생성한 execution role, requirements 파일로 AgentCore Runtime 배포를 구성합니다. 실행 시 Amazon ECR repository를 자동으로 생성하도록 starter toolkit도 구성합니다.

configure 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

discovery_url = input("Enter your discovery URL: ")

client_id = input("Enter your client ID: ")

audience = input("Enter your audience: ")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="my_agent_mcp.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_agent_inbound_identity_okta",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id],
            "allowedAudience": [audience],
        }
    },
)
response

### AgentCore 구성 검토

In [ ]:
!cat .bedrock_agentcore.yaml

#### AgentCore Runtime에 에이전트 실행

Dockerfile이 준비되었으므로 AgentCore Runtime에 에이전트를 실행합니다. 이 과정에서 Amazon ECR repository와 AgentCore Runtime이 생성됩니다.

<figure>
    <img src="images/14.png">
</figure>

In [ ]:
launch_result = agentcore_runtime.launch()
launch_result

#### AgentCore Runtime 상태 확인

AgentCore Runtime을 배포했으므로 배포 상태를 확인합니다.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

### Execution Role에 OAuth Token 읽기 권한 부여

1. <b>Amazon Bedrock AgentCore<b>로 이동합니다.
2. 왼쪽 메뉴에서 <b>Agent Runtime</b>을 클릭합니다.
3. 에이전트를 클릭합니다(예: strands_agent_inbound_identity_okta).
<figure>
    <img src="images/45.png">
</figure>

4. 최신 version을 클릭합니다.
<figure>
    <img src="images/46.png">
</figure>

5. <b>IAM service role</b>을 클릭합니다.
<figure>
    <img src="images/47.png">
</figure>

6. <b>Permissions policies</b>에서 <b>Policy name</b>을 클릭합니다.
<figure>
    <img src="images/48.png">
</figure>

7. 에이전트에 token vault 액세스 권한을 부여하려면 다음 statement를 추가합니다.

		{
			"Sid": "GetResourceOauth2Token",
			"Effect": "Allow",
			"Action": [
				"bedrock-agentcore:GetResourceOauth2Token",
				"secretsmanager:GetSecretValue"
			],
			"Resource": "*"
		}

8. <b>Next</b>를 클릭합니다.
<figure>
    <img src="images/59.png">
</figure>

### 에이전트에 Amazon Bedrock 모델 액세스 권한 부여
1. <b>Amazon Bedrock</b> > <b>Model access</b>로 이동하여 <b>Modify model access</b>를 클릭합니다.
<figure>
    <img src="images/54.png">
</figure>

2. <b>Claude Sonnet 4</b>를 선택하고 <b>Next</b>를 클릭합니다(참고: Strands에서 사용하는 기본 모델은 변경될 수 있습니다). 
<figure>
    <img src="images/55.png">
</figure>

#### 인증 없이 AgentCore Runtime 호출

이제 payload로 AgentCore Runtime을 호출할 수 있습니다. 다음 셀을 실행하면 **"AccessDeniedException: An error occurred (AccessDeniedException) when calling the InvokeAgentRuntime operation: Agent is configured for a different authorization token type".**이라는 오류가 표시됩니다.

<figure>
    <img src="images/15.png">
</figure>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What are the travel plans for customer 123?"})
invoke_response

#### 권한 부여를 사용하여 AgentCore Runtime 호출

올바른 authorization token type으로 에이전트를 호출합니다. 여기서는 Okta access token을 사용합니다.

다음 명령은 다음 단계의 OAuth server에 필요한 **Flask** framework와 **Requests** library를 설치합니다.

In [ ]:
%pip install flask requests

다음 코드는 access token을 가져올 수 있도록 server를 실행합니다. server가 시작되면 link를 클릭하거나 http://127.0.0.1:5000 으로 이동하여 OAuth 흐름을 완료합니다. access token을 가져온 뒤에는 server를 시작할 때와 같은 방식으로 중지합니다.

다음 코드는 아래 작업을 수행합니다.
1. OAuth 인증을 처리하는 작은 web server를 Flask로 생성합니다.
2. 사용자가 ```/login```에 접속하면 Okta login으로 redirect합니다.
3. login 후 server가 `/callback` route를 통해 authorization code를 받습니다.
4. token endpoint에 POST 요청을 보내 code를 access token으로 교환합니다.
5. 성공하면 access token을 session에 저장하고 console에 출력합니다.

**참고**: discovery URL(https://{yoursubdomain}.okta.com/oauth2/default/.well-known/openid-configuration)에서 authorization URL과 token URL을 확인할 수 있습니다.

In [ ]:
import os
import requests
import secrets
from flask import Flask, redirect, request, session

app = Flask(__name__)
app.secret_key = os.urandom(24)

# === 구성 ===
CLIENT_ID = input("Enter your Client ID: ")
CLIENT_SECRET = input("Enter your Client Secret: ")
AUTHORIZATION_BASE_URL = input("Enter your authorization URL: ")
TOKEN_URL = input("Enter your token URL: ")
REDIRECT_URI = "http://127.0.0.1:5000/callback"
SCOPE = "openid email"  # provider에 맞게 조정


# === 1단계: Authorization Server로 redirect ===
@app.route("/")
def home():
    return '<a href="/login">Login with OAuth</a>'


@app.route("/login")
def login():
    state = secrets.token_urlsafe(16)
    session["oauth_state"] = state

    auth_url = (
        f"{AUTHORIZATION_BASE_URL}?response_type=code&client_id={CLIENT_ID}"
        f"&redirect_uri={REDIRECT_URI}&scope={SCOPE}&state={state}"
    )
    return redirect(auth_url)


# === 2단계: callback 처리 및 code를 token으로 교환 ===
@app.route("/callback")
def callback():
    error = request.args.get("error")
    if error:
        return f"Error: {error}"

    code = request.args.get("code")
    if not code:
        return "No code found"

    token_data = {
        "grant_type": "authorization_code",
        "code": code,
        "redirect_uri": REDIRECT_URI,
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    }

    token_response = requests.post(TOKEN_URL, data=token_data)
    token_json = token_response.json()

    if "access_token" in token_json:
        session["access_token"] = token_json["access_token"]
        print("Access Token: " + token_json["access_token"])

        return "Login successful! Access token in logs."
    else:
        return f"Failed to get token: {token_json}"


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000)

access token을 복사하고 다음 코드 섹션에서 요청할 때 입력합니다.

In [ ]:
bearer_token = input("Enter your bearer token: ")
invoke_response = agentcore_runtime.invoke(
    {"prompt": "What flights does customer with user id user-123 have scheduled?"},
    bearer_token=bearer_token,
)
invoke_response

1. AWS Console에서 Amazon Bedrock AgentCore로 이동하고 Build and Deploy 아래의 Agent Runtime을 클릭합니다.
2. 에이전트를 클릭합니다(예: my_agent_mcp).
<figure>
    <img src="images/36.png">
</figure>

3. **Runtime ID**를 복사합니다.
<figure>
    <img src="images/34.png">
</figure>

4. terminal에서 다음 명령을 실행하여 에이전트 log를 가져옵니다. **{myagentruntimeid}**를 에이전트의 **Runtime ID**로 바꿉니다.

```aws logs tail /aws/bedrock-agentcore/runtimes/{myagentruntimeid} --follow```

5. authorization request URL을 확인하여 browser에 붙여넣고 OAuth handshake를 완료합니다.

<figure>
    <img src="images/35.png">
</figure>

## 축하합니다!